1. If you have trained five different models on the exact same training data, and they all achieve 95% precision, is there any chance that you can combine these models to get better results? If so, how? If not, why?

A: Yes you can combine the predictions of each model to potentially achieve better predictions as some models will probably be missing some patterns that the others are capturing resulting in more accurate predictions as a whole.

2. What is the difference between hard and soft voting classifiers?

A: In hard voting, the class with the most predictions gets picked while in soft voting the predictions are weighted by the confidence of the model's predictions to arrive at a final classification.

3. Is it possible to speed up training of a bagging ensemble by distributing it across multiple servers? What about pasting ensembles, boosting ensembles, random forests, or stacking ensembles?

A: Bagging, pasting and RF yes. Boosting is hard to because the next predictor depends on the performance of the last one. The levels of the stacking ensemble can be trained in parallel, but not across levels, because they again depend on the predictions of the previous layer.

4. What is the benefit of out-of-bag evaluation?

A: There's no need to form a separate validation set, which can be helpful especially when data is limited.

5. What makes extra-trees ensembles more random than regular random forests? How can this extra randomness help? Are extra-trees classifiers slower or faster than regular random forests?

A: Extra-trees use a random threshold for each of the features rather than using the optimal one as in regular random forests. The extra randomness will decrease the variance of the predictions (but increase bias), which can help reduce overfitting. Extra-tree classifiers are faster to train than regular random forests as they don't need to find the optimal thresholds for the splits, which is the most compute intensive part of the training.

6. If your AdaBoost ensemble underfits the training data, which hyperparameters should you tweak, and how?

A: I can increase the number of estimators to capture finer patterns in the data. Also, increasing the learning rate can help if the model is slow to converge. Adjusting the hyperparameters of the base model (eg, adding depth to decision trees) will also help with underfitting.

7. If your gradient boosting ensemble overfits the training set, should you increase or decrease the learning rate?

A: I should decrease the learning rate to lessen the contribution of each tree to allow the model to generalize better.

8. Load the MNIST dataset (introduced in Chapter 3), and split it into a training set, a validation set, and a test set (e.g., use 50,000 instances for training, 10,000 for validation, and 10,000 for testing). Then train various classifiers, such as a random forest classifier, an extra-trees classifier, and an SVM classifier. Next, try to combine them into an ensemble that outperforms each individual classifier on the validation set, using soft or hard voting. Once you have found one, try it on the test set. How much better does it perform compared to the individual classifiers?

In [1]:
from sklearn.datasets import fetch_openml
mnist = fetch_openml('mnist_784', as_frame=False)
X, y = mnist.data, mnist.target

In [2]:
from sklearn.model_selection import train_test_split
X_train_temp, X_test, y_train_temp, y_test = train_test_split(X, y, train_size=60000, test_size=10000, random_state=42)

In [3]:
X_train, X_validate, y_train, y_validate = train_test_split(X_train_temp, y_train_temp, 
                                                            train_size=50000, test_size=10000, 
                                                            random_state=42)

In [4]:
print(X_train.shape, X_validate.shape, X_test.shape)

(50000, 784) (10000, 784) (10000, 784)


In [5]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC

rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
et = ExtraTreesClassifier(n_estimators=100, max_depth=5, random_state=42)
svm = SVC(probability=True, C=0.1)

In [6]:
rf.fit(X_train, y_train)
et.fit(X_train, y_train)
svm.fit(X_train, y_train)

SVC(C=0.1, probability=True)

In [7]:
print(f"RF validation accuracy: {rf.score(X_validate, y_validate)}")
print(f"ET validation accuracy: {et.score(X_validate, y_validate)}")
print(f"SVM validation accuracy: {svm.score(X_validate, y_validate)}")

RF validation accuracy: 0.859
ET validation accuracy: 0.8322
SVM validation accuracy: 0.9564


In [8]:
from scipy.stats import mode
import numpy as np

def voting(features):
    # returns (soft_majority, hard_majority)
    
    models = [rf, et, svm]
    probabilities = [model.predict_proba(features) for model in models]
    
    votes = [np.argmax(prob) for prob in probabilities]
    hard_majority = mode(votes).mode
    soft_majority = np.argmax(np.mean(probabilities, axis=0))
    return int(soft_majority), int(hard_majority)


In [9]:
# hard voting
preds_hard = []
for sample in X_validate:
    pred = voting([sample])
    preds_hard.append(pred[1])

In [10]:
# soft voting
preds_soft = []
for sample in X_validate:
    pred = voting([sample])
    preds_soft.append(pred[0])

In [11]:
print(f"Hard voting validation accuracy: {np.mean(preds_hard == y_validate.astype(int)):.4f}")
print(f"Soft voting validation accuracy: {np.mean(preds_soft == y_validate.astype(int)):.4f}")

Hard voting validation accuracy: 0.8835
Soft voting validation accuracy: 0.9537


In [18]:
# test predictions
preds_test = []
for sample in X_test:
    pred = voting([sample])
    preds_test.append(pred[0])

In [19]:
print(f"Soft voting test accuracy: {np.mean(preds_test == y_test.astype(int)):.4f}")

Soft voting test accuracy: 0.9498
